# 🎓 Career Learning Path Recommender SystemA hybrid ML recommender system combining Collaborative Filtering, Content-Based Filtering, and Knowledge Graph approaches.

In [ ]:
# Install dependencies!pip install pandas numpy scikit-learn plotly -q

## 1. Configuration & Imports

In [ ]:
"""Configuration for Career Learning Path Recommender SystemThis module contains all configuration constants and settings used acrossthe recommendation system.Author: RecommenderSystem"""from pathlib import Path# ============================================================================# PATHS# ============================================================================BASE_DIR = Path(__file__).parentDATA_DIR = BASE_DIR / "data"# Data filesCOURSES_FILE = DATA_DIR / "courses.json"USERS_FILE = DATA_DIR / "users.json"INTERACTIONS_FILE = DATA_DIR / "interactions.json"SKILLS_TAXONOMY_FILE = DATA_DIR / "skills_taxonomy.json"CAREER_PATHS_FILE = DATA_DIR / "career_paths.json"# ============================================================================# MODEL PARAMETERS# ============================================================================# Embedding settingsEMBEDDING_MODEL = "all-MiniLM-L6-v2"  # Sentence transformer modelEMBEDDING_DIM = 384# Collaborative filteringCF_MIN_INTERACTIONS = 3  # Minimum interactions to use CFCF_NUM_NEIGHBORS = 10    # Number of similar users/items to considerCF_SIMILARITY_THRESHOLD = 0.1  # Minimum similarity to consider# Content-based filteringCB_SKILL_WEIGHT = 0.7    # Weight for skill matchCB_DIFFICULTY_WEIGHT = 0.2  # Weight for difficulty matchCB_POPULARITY_WEIGHT = 0.1  # Weight for course popularity# Knowledge graphKG_PREREQUISITE_PENALTY = 0.5  # Penalty for missing prerequisitesKG_PATH_BONUS = 0.3  # Bonus for courses on career path# ============================================================================# HYBRID WEIGHTS# ============================================================================# These weights determine how much each recommender contributes# They are dynamically adjusted based on data availabilityHYBRID_WEIGHTS = {    "collaborative": 0.3,    "content_based": 0.4,    "knowledge_graph": 0.3}# Cold-start weights (when user has few interactions)COLD_START_WEIGHTS = {    "collaborative": 0.1,    "content_based": 0.5,    "knowledge_graph": 0.4}# ============================================================================# RECOMMENDATION SETTINGS# ============================================================================DEFAULT_TOP_K = 10  # Default number of recommendationsMAX_TOP_K = 50      # Maximum recommendations to returnMIN_SCORE_THRESHOLD = 0.1  # Minimum score to include in recommendations# Diversity settingsDIVERSITY_WEIGHT = 0.2  # How much to prioritize diversityMAX_SAME_CATEGORY = 3   # Max courses from same category in top-K# ============================================================================# APP SETTINGS# ============================================================================APP_TITLE = "🎓 Career Learning Path Recommender"APP_LAYOUT = "wide"DEBUG_MODE = False# Skill proficiency levelsPROFICIENCY_LEVELS = {    1: "Beginner",    2: "Elementary",     3: "Intermediate",    4: "Advanced",    5: "Expert"}# Course difficulty levelsDIFFICULTY_LEVELS = ["Beginner", "Intermediate", "Advanced", "Expert"]# ============================================================================# DOMAINS# ============================================================================SKILL_DOMAINS = [    "Data Science",    "Machine Learning",    "Cloud Computing",    "DevOps",    "Web Development",    "Mobile Development",    "Cybersecurity",    "Database",    "Programming Languages",    "Soft Skills"]

## 2. Data Models

In [ ]:
"""Data Models for Career Learning Path Recommender SystemThis module defines Pydantic models for all data entities used in the system.Using Pydantic provides:1. Data validation2. Type hints3. Easy serialization/deserialization4. Self-documenting codeAuthor: RecommenderSystem"""from pydantic import BaseModel, Fieldfrom typing import Dict, List, Optionalfrom enum import Enumclass Difficulty(str, Enum):    """Course difficulty levels"""    BEGINNER = "Beginner"    INTERMEDIATE = "Intermediate"    ADVANCED = "Advanced"    EXPERT = "Expert"class LearningStyle(str, Enum):    """User learning style preferences"""    VISUAL = "visual"    HANDS_ON = "hands-on"    READING = "reading"    VIDEO = "video"    PROJECT_BASED = "project-based"    STRUCTURED = "structured"    RESEARCH = "research"class Skill(BaseModel):    """Individual skill definition from taxonomy"""    name: str    level: str  # fundamental, intermediate, advanced, expert    related: List[str] = []    prerequisites: List[str] = []class Course(BaseModel):    """Course in the catalog"""    id: str    title: str    provider: str    skills_taught: List[str]    difficulty: Difficulty    duration_hours: int    rating: float = Field(ge=0, le=5)    num_reviews: int    price: float = Field(ge=0)    category: str    description: str    url: str        def __hash__(self):        return hash(self.id)        def __eq__(self, other):        if isinstance(other, Course):            return self.id == other.id        return Falseclass User(BaseModel):    """User profile"""    id: str    name: str    current_role: str    target_role: str    years_experience: int = Field(ge=0)    current_skills: Dict[str, int]  # skill_id -> proficiency (1-5)    learning_style: LearningStyle    weekly_hours: int = Field(ge=0)    interests: List[str] = []class Interaction(BaseModel):    """User-course interaction"""    user_id: str    course_id: str    enrolled: bool = True    completed: bool = False    rating: Optional[float] = Field(default=None, ge=1, le=5)    timestamp: strclass CareerPath(BaseModel):    """Career role definition"""    name: str    description: str    required_skills: List[str]    nice_to_have: List[str] = []    skill_importance: Dict[str, float] = {}  # skill -> importance weight    salary_range: str = ""    growth_roles: List[str] = []class Recommendation(BaseModel):    """A single course recommendation with explanation"""    course: Course    score: float = Field(ge=0, le=1)    rank: int    reasons: List[str] = []    skill_gaps_addressed: List[str] = []    source: str = "hybrid"  # collaborative, content_based, knowledge_graph, hybridclass RecommendationResult(BaseModel):    """Complete recommendation result for a user"""    user_id: str    target_role: str    recommendations: List[Recommendation]    skill_gap_summary: Dict[str, str] = {}  # skill -> current level description    total_missing_skills: int = 0    estimated_learning_hours: int = 0class SkillGap(BaseModel):    """Skill gap analysis for a user"""    skill_id: str    skill_name: str    current_level: int  # 0 if user doesn't have it    required_level: int  # Based on target role    importance: float  # How important for target role    gap_size: int  # required - current

## 3. Load Data

In [ ]:
import jsonfrom pathlib import Path# Load all data fileswith open('data/courses.json') as f:    courses_data = json.load(f)with open('data/users.json') as f:    users_data = json.load(f)with open('data/interactions.json') as f:    interactions_data = json.load(f)with open('data/skills_taxonomy.json') as f:    skills_data = json.load(f)with open('data/career_paths.json') as f:    career_data = json.load(f)print(f"Loaded {len(courses_data['courses'])} courses")print(f"Loaded {len(users_data['users'])} users")print(f"Loaded {len(interactions_data['interactions'])} interactions")

## 4. Collaborative Filtering Recommender

In [ ]:
"""Collaborative Filtering RecommenderThis module implements collaborative filtering for course recommendations.It uses two approaches:1. User-User Collaborative Filtering: Find similar users and recommend their courses2. Item-Item Collaborative Filtering: Find similar courses based on interaction patternsWHY COLLABORATIVE FILTERING:- Captures "wisdom of the crowd" - professionals who learned X then learned Y- Discovers non-obvious course relationships- Works well when we have interaction dataAuthor: RecommenderSystem"""import numpy as npfrom typing import Dict, List, Tuple, Optionalfrom collections import defaultdictimport loggingfrom models.data_models import Course, User, Interaction, Recommendationfrom config import CF_MIN_INTERACTIONS, CF_NUM_NEIGHBORS, CF_SIMILARITY_THRESHOLDlogger = logging.getLogger(__name__)class CollaborativeFilteringRecommender:    """    Collaborative Filtering Recommender using cosine similarity.        Supports both user-based and item-based collaborative filtering.    """        def __init__(        self,        courses: Dict[str, Course],        users: Dict[str, User],        interactions: List[Interaction]    ):        self.courses = courses        self.users = users        self.interactions = interactions                # Build interaction matrices        self._build_matrices()        def _build_matrices(self):        """Build user-course and course-user matrices."""        # User -> Course -> Score        self.user_course_matrix: Dict[str, Dict[str, float]] = defaultdict(dict)        # Course -> User -> Score          self.course_user_matrix: Dict[str, Dict[str, float]] = defaultdict(dict)        # Track completed courses per user        self.user_completed: Dict[str, set] = defaultdict(set)                for interaction in self.interactions:            # Calculate score based on interaction type            if interaction.rating is not None:                score = interaction.rating / 5.0            elif interaction.completed:                score = 0.8            else:                score = 0.5                        self.user_course_matrix[interaction.user_id][interaction.course_id] = score            self.course_user_matrix[interaction.course_id][interaction.user_id] = score                        if interaction.completed:                self.user_completed[interaction.user_id].add(interaction.course_id)                logger.info(f"Built matrices: {len(self.user_course_matrix)} users, {len(self.course_user_matrix)} courses")        def _cosine_similarity(self, vec1: Dict[str, float], vec2: Dict[str, float]) -> float:        """        Calculate cosine similarity between two sparse vectors.                Args:            vec1: First vector as {key: value} dict            vec2: Second vector as {key: value} dict                    Returns:            Cosine similarity score between 0 and 1        """        # Find common keys        common_keys = set(vec1.keys()) & set(vec2.keys())                if not common_keys:            return 0.0                # Calculate dot product and magnitudes        dot_product = sum(vec1[k] * vec2[k] for k in common_keys)        mag1 = np.sqrt(sum(v**2 for v in vec1.values()))        mag2 = np.sqrt(sum(v**2 for v in vec2.values()))                if mag1 == 0 or mag2 == 0:            return 0.0                return dot_product / (mag1 * mag2)        def _find_similar_users(        self,         user_id: str,         top_n: int = CF_NUM_NEIGHBORS    ) -> List[Tuple[str, float]]:        """        Find users most similar to the given user based on course interactions.                Args:            user_id: Target user ID            top_n: Number of similar users to return                    Returns:            List of (user_id, similarity_score) tuples        """        if user_id not in self.user_course_matrix:            return []                target_vec = self.user_course_matrix[user_id]        similarities = []                for other_id, other_vec in self.user_course_matrix.items():            if other_id == user_id:                continue                        sim = self._cosine_similarity(target_vec, other_vec)            if sim >= CF_SIMILARITY_THRESHOLD:                similarities.append((other_id, sim))                # Sort by similarity (descending) and return top N        similarities.sort(key=lambda x: x[1], reverse=True)        return similarities[:top_n]        def _find_similar_courses(        self,         course_id: str,         top_n: int = CF_NUM_NEIGHBORS    ) -> List[Tuple[str, float]]:        """        Find courses similar to the given course based on user interaction patterns.                Args:            course_id: Target course ID            top_n: Number of similar courses to return                    Returns:            List of (course_id, similarity_score) tuples        """        if course_id not in self.course_user_matrix:            return []                target_vec = self.course_user_matrix[course_id]        similarities = []                for other_id, other_vec in self.course_user_matrix.items():            if other_id == course_id:                continue                        sim = self._cosine_similarity(target_vec, other_vec)            if sim >= CF_SIMILARITY_THRESHOLD:                similarities.append((other_id, sim))                # Sort by similarity (descending) and return top N        similarities.sort(key=lambda x: x[1], reverse=True)        return similarities[:top_n]        def recommend_user_based(        self,         user_id: str,         top_k: int = 10,        exclude_completed: bool = True    ) -> List[Tuple[str, float, str]]:        """        Generate recommendations using user-based collaborative filtering.                Logic: Find users with similar learning patterns and recommend courses        they completed that the target user hasn't taken.                Args:            user_id: Target user ID            top_k: Number of recommendations to return            exclude_completed: Whether to exclude courses user already completed                    Returns:            List of (course_id, score, reason) tuples        """        # Check if user has enough interactions        user_interactions = len(self.user_course_matrix.get(user_id, {}))        if user_interactions < CF_MIN_INTERACTIONS:            logger.info(f"User {user_id} has {user_interactions} interactions (< {CF_MIN_INTERACTIONS})")            return []                # Find similar users        similar_users = self._find_similar_users(user_id)        if not similar_users:            logger.info(f"No similar users found for {user_id}")            return []                # Get courses completed by similar users        user_completed = self.user_completed.get(user_id, set())        course_scores: Dict[str, Tuple[float, str]] = {}                for sim_user_id, similarity in similar_users:            sim_user_completed = self.user_completed.get(sim_user_id, set())                        for course_id in sim_user_completed:                # Skip if user already completed this course                if exclude_completed and course_id in user_completed:                    continue                                # Skip if course not in catalog                if course_id not in self.courses:                    continue                                # Weight by similarity and rating                rating = self.user_course_matrix[sim_user_id].get(course_id, 0.8)                weighted_score = similarity * rating                                if course_id not in course_scores or weighted_score > course_scores[course_id][0]:                    reason = f"Completed by similar user with {similarity:.0%} match"                    course_scores[course_id] = (weighted_score, reason)                # Sort and return top K        sorted_courses = sorted(course_scores.items(), key=lambda x: x[1][0], reverse=True)        return [(cid, score, reason) for cid, (score, reason) in sorted_courses[:top_k]]        def recommend_item_based(        self,         user_id: str,         top_k: int = 10,        exclude_completed: bool = True    ) -> List[Tuple[str, float, str]]:        """        Generate recommendations using item-based collaborative filtering.                Logic: Based on courses the user has completed, find similar courses.                Args:            user_id: Target user ID            top_k: Number of recommendations to return            exclude_completed: Whether to exclude courses user already completed                    Returns:            List of (course_id, score, reason) tuples        """        user_completed = self.user_completed.get(user_id, set())                if not user_completed:            logger.info(f"User {user_id} has no completed courses for item-based CF")            return []                course_scores: Dict[str, Tuple[float, str]] = {}                for completed_course_id in user_completed:            similar_courses = self._find_similar_courses(completed_course_id)                        for similar_id, similarity in similar_courses:                # Skip if user already completed this course                if exclude_completed and similar_id in user_completed:                    continue                                # Skip if course not in catalog                if similar_id not in self.courses:                    continue                                if similar_id not in course_scores or similarity > course_scores[similar_id][0]:                    source_title = self.courses[completed_course_id].title[:30]                    reason = f"Similar to '{source_title}...' ({similarity:.0%} match)"                    course_scores[similar_id] = (similarity, reason)                # Sort and return top K        sorted_courses = sorted(course_scores.items(), key=lambda x: x[1][0], reverse=True)        return [(cid, score, reason) for cid, (score, reason) in sorted_courses[:top_k]]        def recommend(        self,         user_id: str,         top_k: int = 10,        method: str = "combined"    ) -> List[Tuple[str, float, str]]:        """        Generate collaborative filtering recommendations.                Args:            user_id: Target user ID            top_k: Number of recommendations to return            method: "user_based", "item_based", or "combined"                    Returns:            List of (course_id, score, reason) tuples        """        if method == "user_based":            return self.recommend_user_based(user_id, top_k)        elif method == "item_based":            return self.recommend_item_based(user_id, top_k)        else:            # Combine both methods            user_recs = self.recommend_user_based(user_id, top_k)            item_recs = self.recommend_item_based(user_id, top_k)                        # Merge and deduplicate            combined: Dict[str, Tuple[float, str]] = {}                        for course_id, score, reason in user_recs:                combined[course_id] = (score, reason)                        for course_id, score, reason in item_recs:                if course_id in combined:                    # Take max score, combine reasons                    existing_score, existing_reason = combined[course_id]                    if score > existing_score:                        combined[course_id] = (score, reason)                else:                    combined[course_id] = (score, reason)                        # Sort and return top K            sorted_courses = sorted(combined.items(), key=lambda x: x[1][0], reverse=True)            return [(cid, score, reason) for cid, (score, reason) in sorted_courses[:top_k]]        def get_user_stats(self, user_id: str) -> Dict:        """Get statistics about user's interaction data for debugging."""        return {            "total_interactions": len(self.user_course_matrix.get(user_id, {})),            "completed_courses": len(self.user_completed.get(user_id, set())),            "similar_users_count": len(self._find_similar_users(user_id)),            "has_enough_data": len(self.user_course_matrix.get(user_id, {})) >= CF_MIN_INTERACTIONS        }

## 5. Content-Based Recommender

In [ ]:
"""Content-Based Filtering RecommenderThis module implements content-based filtering for course recommendations.It recommends courses based on:1. Skill gap analysis (missing skills for target role)2. Skill matching (courses that teach needed skills)3. User preferences (difficulty level, interests)WHY CONTENT-BASED FILTERING:- Works with new users (no cold-start problem)- Directly addresses skill gaps- Explainable recommendations ("this course teaches X which you need")Author: RecommenderSystem"""from typing import Dict, List, Tuple, Setimport loggingfrom models.data_models import Course, User, CareerPath, SkillGap, Difficultyfrom config import CB_SKILL_WEIGHT, CB_DIFFICULTY_WEIGHT, CB_POPULARITY_WEIGHTlogger = logging.getLogger(__name__)class ContentBasedRecommender:    """    Content-Based Recommender using skill matching and gap analysis.    """        def __init__(        self,        courses: Dict[str, Course],        users: Dict[str, User],        career_paths: Dict[str, CareerPath],        skills_taxonomy: Dict    ):        self.courses = courses        self.users = users        self.career_paths = career_paths        self.skills = skills_taxonomy['skills']                # Build course skill index for fast lookup        self._build_skill_index()        def _build_skill_index(self):        """Build inverted index: skill -> courses that teach it."""        self.skill_to_courses: Dict[str, List[str]] = {}                for course_id, course in self.courses.items():            for skill in course.skills_taught:                if skill not in self.skill_to_courses:                    self.skill_to_courses[skill] = []                self.skill_to_courses[skill].append(course_id)                logger.info(f"Built skill index with {len(self.skill_to_courses)} skills")        def analyze_skill_gap(        self,         user: User,         career_path: CareerPath    ) -> List[SkillGap]:        """        Analyze the skill gap between user's current skills and target role requirements.                Args:            user: User profile            career_path: Target career path                    Returns:            List of SkillGap objects, sorted by importance        """        skill_gaps = []                # Get required skills with importance weights        skill_importance = career_path.skill_importance        all_required = set(career_path.required_skills + career_path.nice_to_have)                for skill_id in all_required:            current_level = user.current_skills.get(skill_id, 0)                        # Determine required level based on importance            importance = skill_importance.get(skill_id, 0.5)            if importance >= 0.9:                required_level = 5            elif importance >= 0.7:                required_level = 4            elif importance >= 0.5:                required_level = 3            else:                required_level = 2                        # Only add as gap if there's actually a gap            if current_level < required_level:                skill_name = self.skills.get(skill_id, {}).get('name', skill_id)                gap = SkillGap(                    skill_id=skill_id,                    skill_name=skill_name,                    current_level=current_level,                    required_level=required_level,                    importance=importance,                    gap_size=required_level - current_level                )                skill_gaps.append(gap)                # Sort by importance (descending)        skill_gaps.sort(key=lambda x: (x.importance, x.gap_size), reverse=True)                logger.info(f"Found {len(skill_gaps)} skill gaps for user {user.id}")        return skill_gaps        def _get_difficulty_score(self, user: User, course: Course) -> float:        """        Calculate difficulty appropriateness score.                Logic:        - If user is beginner and course is advanced, lower score        - If user is advanced and course is beginner, lower score        - Sweet spot: course is one level above user's average        """        # Estimate user level from average skill proficiency        if user.current_skills:            avg_proficiency = sum(user.current_skills.values()) / len(user.current_skills)        else:            avg_proficiency = 1.0                # Map user proficiency to difficulty        user_level = min(3, int(avg_proficiency))  # 0, 1, 2, 3                difficulty_order = {            Difficulty.BEGINNER: 0,            Difficulty.INTERMEDIATE: 1,            Difficulty.ADVANCED: 2,            Difficulty.EXPERT: 3        }        course_level = difficulty_order.get(course.difficulty, 1)                # Ideal: course is same level or one level up        level_diff = abs(course_level - user_level)                if course_level == user_level + 1:            return 1.0  # Ideal stretch        elif course_level == user_level:            return 0.9  # Good match        elif level_diff == 1:            return 0.7  # Acceptable        elif level_diff == 2:            return 0.4  # Less ideal        else:            return 0.2  # Too far        def _get_popularity_score(self, course: Course) -> float:        """Normalize popularity score based on reviews and rating."""        # Normalize reviews (log scale)        import math        review_score = min(1.0, math.log10(course.num_reviews + 1) / 5)        rating_score = course.rating / 5.0                return 0.5 * review_score + 0.5 * rating_score        def recommend(        self,         user_id: str,         top_k: int = 10,        exclude_user_skills: bool = True    ) -> List[Tuple[str, float, str, List[str]]]:        """        Generate content-based recommendations for a user.                Args:            user_id: Target user ID            top_k: Number of recommendations to return            exclude_user_skills: Whether to exclude courses teaching only skills user has                    Returns:            List of (course_id, score, reason, skills_addressed) tuples        """        if user_id not in self.users:            logger.warning(f"User {user_id} not found")            return []                user = self.users[user_id]                # Get career path        if user.target_role not in self.career_paths:            logger.warning(f"Career path {user.target_role} not found")            return []                career_path = self.career_paths[user.target_role]                # Analyze skill gaps        skill_gaps = self.analyze_skill_gap(user, career_path)        missing_skills = {gap.skill_id: gap for gap in skill_gaps}                if not missing_skills:            logger.info(f"User {user_id} has no skill gaps for {user.target_role}")            return []                # Score each course        course_scores: Dict[str, Tuple[float, str, List[str]]] = {}                for course_id, course in self.courses.items():            # Find skills this course teaches that user needs            skills_addressed = [                skill for skill in course.skills_taught                 if skill in missing_skills            ]                        if not skills_addressed:                continue                        # Calculate skill match score            skill_score = 0.0            for skill_id in skills_addressed:                gap = missing_skills[skill_id]                # Weight by importance and gap size                skill_score += gap.importance * (gap.gap_size / 5.0)                        # Normalize by number of skills            skill_score = min(1.0, skill_score / len(skills_addressed))                        # Calculate other scores            difficulty_score = self._get_difficulty_score(user, course)            popularity_score = self._get_popularity_score(course)                        # Weighted combination            total_score = (                CB_SKILL_WEIGHT * skill_score +                CB_DIFFICULTY_WEIGHT * difficulty_score +                CB_POPULARITY_WEIGHT * popularity_score            )                        # Generate reason            skill_names = [missing_skills[s].skill_name for s in skills_addressed[:3]]            if len(skills_addressed) > 3:                reason = f"Teaches {', '.join(skill_names)} +{len(skills_addressed)-3} more needed skills"            else:                reason = f"Teaches {', '.join(skill_names)} which you need for {career_path.name}"                        course_scores[course_id] = (total_score, reason, skills_addressed)                # Sort and return top K        sorted_courses = sorted(course_scores.items(), key=lambda x: x[1][0], reverse=True)        return [(cid, score, reason, skills) for cid, (score, reason, skills) in sorted_courses[:top_k]]        def get_skill_gap_summary(self, user_id: str) -> Dict[str, List[SkillGap]]:        """        Get categorized skill gaps for a user.                Returns:            Dictionary with 'critical', 'important', 'nice_to_have' skill gap lists        """        if user_id not in self.users:            return {}                user = self.users[user_id]                if user.target_role not in self.career_paths:            return {}                career_path = self.career_paths[user.target_role]        skill_gaps = self.analyze_skill_gap(user, career_path)                categorized = {            'critical': [],            'important': [],            'nice_to_have': []        }                for gap in skill_gaps:            if gap.importance >= 0.85:                categorized['critical'].append(gap)            elif gap.importance >= 0.6:                categorized['important'].append(gap)            else:                categorized['nice_to_have'].append(gap)                return categorized

## 6. Knowledge Graph Recommender

In [ ]:
"""Knowledge Graph RecommenderThis module implements career-path-aware recommendations using a knowledge graph approach.It considers:1. Skill prerequisites (learn X before Y)2. Career path progression (skills needed for target role)3. Learning path sequencing (ordered course recommendations)WHY KNOWLEDGE GRAPH:- Ensures logical learning progression- Respects skill dependencies- Creates coherent learning paths, not just isolated coursesAuthor: RecommenderSystem"""from typing import Dict, List, Tuple, Setfrom collections import defaultdictimport loggingfrom models.data_models import Course, User, CareerPath, Difficultyfrom config import KG_PREREQUISITE_PENALTY, KG_PATH_BONUSlogger = logging.getLogger(__name__)class KnowledgeGraphRecommender:    """    Knowledge Graph Recommender using skill relationships and career paths.        This recommender builds a graph of skill relationships and uses it to:    1. Identify optimal learning sequences    2. Penalize courses with unmet prerequisites    3. Prioritize courses on the career path    """        def __init__(        self,        courses: Dict[str, Course],        users: Dict[str, User],        career_paths: Dict[str, CareerPath],        skills_taxonomy: Dict    ):        self.courses = courses        self.users = users        self.career_paths = career_paths        self.skills = skills_taxonomy['skills']                # Build skill graph        self._build_skill_graph()            def _build_skill_graph(self):        """Build directed graph of skill prerequisites and relationships."""        # Prerequisites: skill -> list of prerequisite skills        self.prerequisites: Dict[str, List[str]] = {}        # Dependents: skill -> list of skills that depend on it        self.dependents: Dict[str, List[str]] = defaultdict(list)        # Skill levels        self.skill_levels: Dict[str, str] = {}                for skill_id, skill_data in self.skills.items():            prereqs = skill_data.get('prerequisites', [])            self.prerequisites[skill_id] = prereqs            self.skill_levels[skill_id] = skill_data.get('level', 'intermediate')                        for prereq in prereqs:                self.dependents[prereq].append(skill_id)                logger.info(f"Built skill graph with {len(self.prerequisites)} skills")        def _get_missing_prerequisites(        self,         skill_id: str,         user_skills: Set[str]    ) -> List[str]:        """        Find prerequisites for a skill that the user doesn't have.                Args:            skill_id: Target skill            user_skills: Set of skills user already has                    Returns:            List of missing prerequisite skill IDs        """        prereqs = self.prerequisites.get(skill_id, [])        return [p for p in prereqs if p not in user_skills]        def _get_skill_depth(self, skill_id: str, memo: Dict[str, int] = None) -> int:        """        Calculate the depth of a skill in the prerequisite graph.        Depth 0 = no prerequisites, higher = more advanced.        """        if memo is None:            memo = {}                if skill_id in memo:            return memo[skill_id]                prereqs = self.prerequisites.get(skill_id, [])        if not prereqs:            memo[skill_id] = 0            return 0                max_prereq_depth = max(self._get_skill_depth(p, memo) for p in prereqs)        memo[skill_id] = max_prereq_depth + 1        return memo[skill_id]        def _get_learning_path_order(        self,         target_skills: List[str],         user_skills: Set[str]    ) -> List[str]:        """        Order skills by their logical learning sequence.                Uses topological sort considering prerequisites.                Args:            target_skills: Skills to learn            user_skills: Skills user already has                    Returns:            Ordered list of skills to learn        """        # Expand to include prerequisites        all_skills_needed = set()        stack = list(target_skills)                while stack:            skill = stack.pop()            if skill in user_skills or skill in all_skills_needed:                continue                        all_skills_needed.add(skill)            prereqs = self.prerequisites.get(skill, [])            stack.extend(prereqs)                # Sort by depth (learn fundamental skills first)        depth_memo = {}        skills_with_depth = [            (skill, self._get_skill_depth(skill, depth_memo))            for skill in all_skills_needed        ]        skills_with_depth.sort(key=lambda x: x[1])                return [skill for skill, _ in skills_with_depth]        def _calculate_prerequisite_score(        self,         course: Course,         user_skills: Set[str]    ) -> Tuple[float, List[str]]:        """        Calculate how ready the user is to take this course.                Returns:            Tuple of (readiness_score, missing_prerequisites)        """        missing_prereqs = set()                for skill in course.skills_taught:            missing = self._get_missing_prerequisites(skill, user_skills)            missing_prereqs.update(missing)                if not missing_prereqs:            return 1.0, []                # Penalize based on number of missing prerequisites        penalty = KG_PREREQUISITE_PENALTY * len(missing_prereqs)        score = max(0.1, 1.0 - penalty)                return score, list(missing_prereqs)        def _calculate_path_alignment_score(        self,         course: Course,         career_path: CareerPath    ) -> float:        """        Calculate how well course aligns with career path.                Returns:            Score between 0 and 1        """        required_skills = set(career_path.required_skills)        nice_to_have = set(career_path.nice_to_have)        course_skills = set(course.skills_taught)                required_overlap = len(course_skills & required_skills)        nice_overlap = len(course_skills & nice_to_have)                if not course_skills:            return 0.0                # Weight required skills higher        score = (required_overlap * 1.0 + nice_overlap * 0.5) / len(course_skills)        return min(1.0, score + KG_PATH_BONUS if required_overlap > 0 else score)        def recommend(        self,         user_id: str,         top_k: int = 10    ) -> List[Tuple[str, float, str, int]]:        """        Generate knowledge-graph-aware recommendations.                Args:            user_id: Target user ID            top_k: Number of recommendations to return                    Returns:            List of (course_id, score, reason, sequence_order) tuples        """        if user_id not in self.users:            logger.warning(f"User {user_id} not found")            return []                user = self.users[user_id]        user_skills = set(user.current_skills.keys())                if user.target_role not in self.career_paths:            logger.warning(f"Career path {user.target_role} not found")            return []                career_path = self.career_paths[user.target_role]                # Get ordered learning path        target_skills = career_path.required_skills + career_path.nice_to_have        ordered_skills = self._get_learning_path_order(target_skills, user_skills)                # Create skill priority map (earlier in path = higher priority)        skill_priority = {skill: i for i, skill in enumerate(ordered_skills)}                # Score each course        course_scores: Dict[str, Tuple[float, str, int]] = {}                for course_id, course in self.courses.items():            # Check if course teaches any needed skills            teaches_needed = [s for s in course.skills_taught if s in skill_priority]            if not teaches_needed:                continue                        # Calculate scores            prereq_score, missing = self._calculate_prerequisite_score(course, user_skills)            path_score = self._calculate_path_alignment_score(course, career_path)                        # Prioritize courses for earlier skills in the learning path            earliest_skill = min(skill_priority.get(s, 999) for s in teaches_needed)            sequence_score = 1.0 / (1 + earliest_skill * 0.1)  # Higher for earlier skills                        # Combined score            total_score = 0.4 * prereq_score + 0.3 * path_score + 0.3 * sequence_score                        # Generate reason            if missing:                missing_names = [self.skills.get(m, {}).get('name', m) for m in missing[:2]]                reason = f"Learn after: {', '.join(missing_names)}"            else:                taught_names = [self.skills.get(s, {}).get('name', s) for s in teaches_needed[:2]]                reason = f"Next step: teaches {', '.join(taught_names)}"                        course_scores[course_id] = (total_score, reason, earliest_skill)                # Sort by score, then by sequence order        sorted_courses = sorted(            course_scores.items(),             key=lambda x: (-x[1][0], x[1][2])  # Score desc, then sequence asc        )                return [(cid, score, reason, seq) for cid, (score, reason, seq) in sorted_courses[:top_k]]        def get_learning_path(        self,         user_id: str,         max_courses: int = 10    ) -> List[Course]:        """        Generate an ordered learning path for the user.                Returns courses in the recommended order to take them.        """        recommendations = self.recommend(user_id, max_courses)                # Sort by sequence order        sorted_recs = sorted(recommendations, key=lambda x: x[3])                return [self.courses[course_id] for course_id, _, _, _ in sorted_recs]        def get_next_skills(self, user_id: str, count: int = 5) -> List[Dict]:        """        Get the next skills the user should learn in order.                Returns:            List of skill dictionaries with info        """        if user_id not in self.users:            return []                user = self.users[user_id]        user_skills = set(user.current_skills.keys())                if user.target_role not in self.career_paths:            return []                career_path = self.career_paths[user.target_role]        target_skills = career_path.required_skills + career_path.nice_to_have        ordered_skills = self._get_learning_path_order(target_skills, user_skills)                result = []        for skill_id in ordered_skills[:count]:            skill_data = self.skills.get(skill_id, {})            missing_prereqs = self._get_missing_prerequisites(skill_id, user_skills)                        result.append({                'skill_id': skill_id,                'name': skill_data.get('name', skill_id),                'level': skill_data.get('level', 'intermediate'),                'missing_prerequisites': missing_prereqs,                'ready_to_learn': len(missing_prereqs) == 0            })                return result

## 7. Hybrid Recommender

In [ ]:
"""Hybrid Recommender SystemThis module combines multiple recommendation strategies into a unified hybrid system.It intelligently weights different recommenders based on:1. Data availability (cold-start handling)2. User context (new vs. experienced user)3. Recommendation quality signalsWHY HYBRID:- Best of all worlds: collaborative + content + knowledge graph- Graceful degradation when data is sparse- More diverse and robust recommendationsAuthor: RecommenderSystem"""from typing import Dict, List, Tuple, Optionalfrom collections import defaultdictimport loggingfrom models.data_models import (    Course, User, Interaction, CareerPath,     Recommendation, RecommendationResult, SkillGap)from recommenders.collaborative import CollaborativeFilteringRecommenderfrom recommenders.content_based import ContentBasedRecommenderfrom recommenders.knowledge_graph import KnowledgeGraphRecommenderfrom config import (    HYBRID_WEIGHTS, COLD_START_WEIGHTS,     CF_MIN_INTERACTIONS, MIN_SCORE_THRESHOLD,    DIVERSITY_WEIGHT, MAX_SAME_CATEGORY)logger = logging.getLogger(__name__)class HybridRecommender:    """    Hybrid Recommender that combines collaborative, content-based,     and knowledge graph approaches.    """        def __init__(        self,        courses: Dict[str, Course],        users: Dict[str, User],        interactions: List[Interaction],        career_paths: Dict[str, CareerPath],        skills_taxonomy: Dict    ):        self.courses = courses        self.users = users        self.interactions = interactions        self.career_paths = career_paths        self.skills_taxonomy = skills_taxonomy                # Initialize individual recommenders        self.cf_recommender = CollaborativeFilteringRecommender(            courses, users, interactions        )        self.cb_recommender = ContentBasedRecommender(            courses, users, career_paths, skills_taxonomy        )        self.kg_recommender = KnowledgeGraphRecommender(            courses, users, career_paths, skills_taxonomy        )                logger.info("Hybrid recommender initialized with all components")        def _get_weights(self, user_id: str) -> Dict[str, float]:        """        Determine weights for each recommender based on user data availability.                Cold-start users get more weight on content-based and knowledge graph.        Users with history get more weight on collaborative filtering.        """        cf_stats = self.cf_recommender.get_user_stats(user_id)                if cf_stats['has_enough_data']:            # User has enough interaction data            weights = HYBRID_WEIGHTS.copy()                        # Boost CF if user has many similar users            if cf_stats['similar_users_count'] >= 5:                weights['collaborative'] += 0.1                weights['content_based'] -= 0.05                weights['knowledge_graph'] -= 0.05        else:            # Cold-start: rely more on content and knowledge graph            weights = COLD_START_WEIGHTS.copy()                # Normalize weights        total = sum(weights.values())        return {k: v / total for k, v in weights.items()}        def _merge_recommendations(        self,        cf_recs: List[Tuple[str, float, str]],        cb_recs: List[Tuple[str, float, str, List[str]]],        kg_recs: List[Tuple[str, float, str, int]],        weights: Dict[str, float],        top_k: int    ) -> List[Dict]:        """        Merge recommendations from all sources with weighted scoring.                Returns:            List of merged recommendation dictionaries        """        merged: Dict[str, Dict] = {}                # Process collaborative filtering recommendations        for course_id, score, reason in cf_recs:            if course_id not in merged:                merged[course_id] = {                    'course_id': course_id,                    'scores': {},                    'reasons': [],                    'skills_addressed': [],                    'sources': []                }            merged[course_id]['scores']['collaborative'] = score            merged[course_id]['reasons'].append(f"[CF] {reason}")            merged[course_id]['sources'].append('collaborative')                # Process content-based recommendations        for course_id, score, reason, skills in cb_recs:            if course_id not in merged:                merged[course_id] = {                    'course_id': course_id,                    'scores': {},                    'reasons': [],                    'skills_addressed': [],                    'sources': []                }            merged[course_id]['scores']['content_based'] = score            merged[course_id]['reasons'].append(f"[CB] {reason}")            merged[course_id]['skills_addressed'].extend(skills)            merged[course_id]['sources'].append('content_based')                # Process knowledge graph recommendations        for course_id, score, reason, seq in kg_recs:            if course_id not in merged:                merged[course_id] = {                    'course_id': course_id,                    'scores': {},                    'reasons': [],                    'skills_addressed': [],                    'sources': []                }            merged[course_id]['scores']['knowledge_graph'] = score            merged[course_id]['reasons'].append(f"[KG] {reason}")            merged[course_id]['sources'].append('knowledge_graph')                # Calculate final weighted scores        for course_id, data in merged.items():            weighted_score = 0.0            for source, weight in weights.items():                if source in data['scores']:                    weighted_score += weight * data['scores'][source]            data['final_score'] = weighted_score                        # Deduplicate skills_addressed            data['skills_addressed'] = list(set(data['skills_addressed']))                # Sort by final score        sorted_recs = sorted(merged.values(), key=lambda x: x['final_score'], reverse=True)                # Apply diversity filter        return self._apply_diversity(sorted_recs, top_k)        def _apply_diversity(        self,         recommendations: List[Dict],         top_k: int    ) -> List[Dict]:        """        Apply diversity filtering to avoid too many courses from same category.        """        result = []        category_counts: Dict[str, int] = defaultdict(int)                for rec in recommendations:            course = self.courses.get(rec['course_id'])            if not course:                continue                        category = course.category                        # Apply category limit            if category_counts[category] >= MAX_SAME_CATEGORY:                # Still add if score is very high                if rec['final_score'] > 0.8:                    result.append(rec)                continue                        category_counts[category] += 1            result.append(rec)                        if len(result) >= top_k:                break                return result[:top_k]        def recommend(        self,         user_id: str,         top_k: int = 10    ) -> RecommendationResult:        """        Generate hybrid recommendations for a user.                Args:            user_id: Target user ID            top_k: Number of recommendations to return                    Returns:            RecommendationResult with ranked courses and explanations        """        if user_id not in self.users:            logger.warning(f"User {user_id} not found")            return RecommendationResult(                user_id=user_id,                target_role="unknown",                recommendations=[],                total_missing_skills=0,                estimated_learning_hours=0            )                user = self.users[user_id]                # Get dynamic weights        weights = self._get_weights(user_id)        logger.info(f"Using weights for {user_id}: {weights}")                # Get recommendations from each source        # Request more than needed to have options after diversity filtering        request_k = top_k * 2                cf_recs = self.cf_recommender.recommend(user_id, request_k)        cb_recs = self.cb_recommender.recommend(user_id, request_k)        kg_recs = self.kg_recommender.recommend(user_id, request_k)                logger.info(f"Got {len(cf_recs)} CF, {len(cb_recs)} CB, {len(kg_recs)} KG recommendations")                # Merge recommendations        merged = self._merge_recommendations(cf_recs, cb_recs, kg_recs, weights, top_k)                # Filter by minimum score        filtered = [r for r in merged if r['final_score'] >= MIN_SCORE_THRESHOLD]                # Build Recommendation objects        recommendations = []        total_hours = 0                for rank, rec_data in enumerate(filtered, 1):            course = self.courses.get(rec_data['course_id'])            if not course:                continue                        recommendation = Recommendation(                course=course,                score=rec_data['final_score'],                rank=rank,                reasons=rec_data['reasons'][:3],  # Top 3 reasons                skill_gaps_addressed=rec_data['skills_addressed'][:5],                source='+'.join(set(rec_data['sources']))            )            recommendations.append(recommendation)            total_hours += course.duration_hours                # Get skill gap summary        skill_gaps = self.cb_recommender.get_skill_gap_summary(user_id)        total_missing = sum(len(gaps) for gaps in skill_gaps.values())                # Format skill gap summary        gap_summary = {}        for category, gaps in skill_gaps.items():            for gap in gaps[:3]:  # Top 3 per category                level_desc = f"Level {gap.current_level}/5 → {gap.required_level}/5 needed"                gap_summary[gap.skill_name] = level_desc                return RecommendationResult(            user_id=user_id,            target_role=user.target_role,            recommendations=recommendations,            skill_gap_summary=gap_summary,            total_missing_skills=total_missing,            estimated_learning_hours=total_hours        )        def get_learning_path(        self,         user_id: str,         max_courses: int = 10    ) -> List[Course]:        """        Get ordered learning path using knowledge graph recommender.        """        return self.kg_recommender.get_learning_path(user_id, max_courses)        def get_skill_gaps(self, user_id: str) -> Dict[str, List[SkillGap]]:        """        Get categorized skill gaps for a user.        """        return self.cb_recommender.get_skill_gap_summary(user_id)        def get_next_skills(self, user_id: str, count: int = 5) -> List[Dict]:        """        Get next skills to learn in order.        """        return self.kg_recommender.get_next_skills(user_id, count)        def explain_recommendation(        self,         user_id: str,         course_id: str    ) -> Dict:        """        Get detailed explanation for why a course was recommended.        """        if course_id not in self.courses:            return {"error": "Course not found"}                course = self.courses[course_id]        user = self.users.get(user_id)                if not user:            return {"error": "User not found"}                # Get individual scores        cf_recs = self.cf_recommender.recommend(user_id, 50)        cb_recs = self.cb_recommender.recommend(user_id, 50)        kg_recs = self.kg_recommender.recommend(user_id, 50)                cf_score = next((s for c, s, _ in cf_recs if c == course_id), None)        cb_data = next(((s, r, sk) for c, s, r, sk in cb_recs if c == course_id), None)        kg_data = next(((s, r, seq) for c, s, r, seq in kg_recs if c == course_id), None)                weights = self._get_weights(user_id)                explanation = {            "course": {                "id": course.id,                "title": course.title,                "skills_taught": course.skills_taught,                "difficulty": course.difficulty.value,                "duration_hours": course.duration_hours            },            "user_context": {                "current_skills": list(user.current_skills.keys()),                "target_role": user.target_role,                "years_experience": user.years_experience            },            "scores": {                "collaborative_filtering": {                    "score": cf_score,                    "weight": weights['collaborative'],                    "reason": "Based on similar users' learning patterns"                },                "content_based": {                    "score": cb_data[0] if cb_data else None,                    "weight": weights['content_based'],                    "skills_addressed": cb_data[2] if cb_data else [],                    "reason": cb_data[1] if cb_data else "Not recommended by this method"                },                "knowledge_graph": {                    "score": kg_data[0] if kg_data else None,                    "weight": weights['knowledge_graph'],                    "sequence_position": kg_data[2] if kg_data else None,                    "reason": kg_data[1] if kg_data else "Not recommended by this method"                }            }        }                return explanation

## 8. Evaluation Metrics

In [ ]:
"""Evaluation Metrics for Recommender SystemThis module implements standard recommendation metrics to evaluate system quality.Metrics include:- Precision@K: Relevant items in top-K- Recall@K: Coverage of relevant items  - NDCG@K: Normalized Discounted Cumulative Gain (ranking quality)- Coverage: Percentage of catalog recommended- Diversity: Intra-list diversityAuthor: RecommenderSystem"""import numpy as npfrom typing import Dict, List, Setfrom collections import defaultdictimport loggingfrom models.data_models import Course, User, Interaction, Recommendationlogger = logging.getLogger(__name__)def precision_at_k(    recommended: List[str],     relevant: Set[str],     k: int) -> float:    """    Calculate Precision@K: proportion of recommended items that are relevant.        Args:        recommended: List of recommended course IDs (ordered)        relevant: Set of relevant course IDs        k: Number of top recommendations to consider            Returns:        Precision@K score between 0 and 1    """    if k <= 0:        return 0.0        top_k = recommended[:k]    if not top_k:        return 0.0        relevant_in_top_k = len(set(top_k) & relevant)    return relevant_in_top_k / kdef recall_at_k(    recommended: List[str],     relevant: Set[str],     k: int) -> float:    """    Calculate Recall@K: proportion of relevant items that are recommended.        Args:        recommended: List of recommended course IDs (ordered)        relevant: Set of relevant course IDs        k: Number of top recommendations to consider            Returns:        Recall@K score between 0 and 1    """    if not relevant:        return 0.0        top_k = recommended[:k]    relevant_in_top_k = len(set(top_k) & relevant)    return relevant_in_top_k / len(relevant)def dcg_at_k(scores: List[float], k: int) -> float:    """    Calculate Discounted Cumulative Gain at K.        Args:        scores: List of relevance scores (ordered by recommendation rank)        k: Number of top items to consider            Returns:        DCG@K value    """    scores = scores[:k]    if not scores:        return 0.0        # DCG = sum(rel_i / log2(i+2)) for i in 0..k-1    gains = [score / np.log2(i + 2) for i, score in enumerate(scores)]    return sum(gains)def ndcg_at_k(    recommended: List[str],    relevance_scores: Dict[str, float],    k: int) -> float:    """    Calculate Normalized Discounted Cumulative Gain at K.        Args:        recommended: List of recommended course IDs (ordered)        relevance_scores: Dict mapping course_id to relevance score        k: Number of top recommendations to consider            Returns:        NDCG@K score between 0 and 1    """    # Get relevance scores for recommended items    rec_scores = [relevance_scores.get(cid, 0.0) for cid in recommended[:k]]        # Calculate DCG    dcg = dcg_at_k(rec_scores, k)        # Calculate ideal DCG (best possible ordering)    ideal_scores = sorted(relevance_scores.values(), reverse=True)[:k]    idcg = dcg_at_k(ideal_scores, k)        if idcg == 0:        return 0.0        return dcg / idcgdef catalog_coverage(    all_recommendations: List[List[str]],    catalog_size: int) -> float:    """    Calculate catalog coverage: percentage of items ever recommended.        Args:        all_recommendations: List of recommendation lists (one per user)        catalog_size: Total number of items in catalog            Returns:        Coverage percentage between 0 and 1    """    if catalog_size == 0:        return 0.0        unique_recommended = set()    for recs in all_recommendations:        unique_recommended.update(recs)        return len(unique_recommended) / catalog_sizedef intra_list_diversity(    recommended: List[str],    courses: Dict[str, Course]) -> float:    """    Calculate intra-list diversity based on category variety.        Args:        recommended: List of recommended course IDs        courses: Course catalog            Returns:        Diversity score between 0 and 1    """    if len(recommended) <= 1:        return 1.0  # Single item is maximally diverse with itself        categories = []    for cid in recommended:        if cid in courses:            categories.append(courses[cid].category)        if not categories:        return 0.0        # Diversity = unique categories / total items (normalized)    unique_categories = len(set(categories))    return unique_categories / len(categories)class RecommenderEvaluator:    """    Evaluator class for comprehensive recommendation system evaluation.    """        def __init__(        self,        courses: Dict[str, Course],        users: Dict[str, User],        interactions: List[Interaction]    ):        self.courses = courses        self.users = users        self.interactions = interactions                # Build ground truth: courses that users completed with high ratings        self._build_ground_truth()        def _build_ground_truth(self):        """Build ground truth relevant sets for each user."""        self.ground_truth: Dict[str, Set[str]] = defaultdict(set)        self.user_ratings: Dict[str, Dict[str, float]] = defaultdict(dict)                for interaction in self.interactions:            if interaction.completed:                self.ground_truth[interaction.user_id].add(interaction.course_id)                                if interaction.rating:                    self.user_ratings[interaction.user_id][interaction.course_id] = interaction.rating / 5.0                else:                    self.user_ratings[interaction.user_id][interaction.course_id] = 0.8        def evaluate_recommendations(        self,        user_id: str,        recommendations: List[str],        k_values: List[int] = [5, 10]    ) -> Dict[str, float]:        """        Evaluate recommendations for a single user.                Args:            user_id: User ID            recommendations: List of recommended course IDs            k_values: Values of K for @K metrics                    Returns:            Dictionary of metric names to values        """        relevant = self.ground_truth.get(user_id, set())        ratings = self.user_ratings.get(user_id, {})                metrics = {}                for k in k_values:            metrics[f'precision@{k}'] = precision_at_k(recommendations, relevant, k)            metrics[f'recall@{k}'] = recall_at_k(recommendations, relevant, k)            metrics[f'ndcg@{k}'] = ndcg_at_k(recommendations, ratings, k)                metrics['diversity'] = intra_list_diversity(recommendations, self.courses)                return metrics        def evaluate_system(        self,        recommender,        user_ids: List[str] = None,        top_k: int = 10    ) -> Dict[str, float]:        """        Evaluate recommender system across multiple users.                Args:            recommender: Recommender object with recommend() method            user_ids: List of user IDs to evaluate (default: all)            top_k: Number of recommendations per user                    Returns:            Dictionary of aggregated metrics        """        if user_ids is None:            user_ids = list(self.users.keys())                all_metrics = defaultdict(list)        all_recommendations = []                for user_id in user_ids:            # Skip users with no ground truth            if user_id not in self.ground_truth:                continue                        # Get recommendations            result = recommender.recommend(user_id, top_k)            rec_ids = [r.course.id for r in result.recommendations]            all_recommendations.append(rec_ids)                        # Evaluate            user_metrics = self.evaluate_recommendations(user_id, rec_ids)            for metric_name, value in user_metrics.items():                all_metrics[metric_name].append(value)                # Aggregate metrics (mean)        aggregated = {}        for metric_name, values in all_metrics.items():            if values:                aggregated[f'mean_{metric_name}'] = np.mean(values)                aggregated[f'std_{metric_name}'] = np.std(values)                # Add coverage        aggregated['catalog_coverage'] = catalog_coverage(            all_recommendations,             len(self.courses)        )                return aggregated        def compare_recommenders(        self,        recommenders: Dict[str, object],        user_ids: List[str] = None,        top_k: int = 10    ) -> Dict[str, Dict[str, float]]:        """        Compare multiple recommenders side by side.                Args:            recommenders: Dictionary of {name: recommender_object}            user_ids: List of user IDs to evaluate            top_k: Number of recommendations per user                    Returns:            Dictionary of {recommender_name: metrics}        """        results = {}                for name, recommender in recommenders.items():            logger.info(f"Evaluating {name}...")            results[name] = self.evaluate_system(recommender, user_ids, top_k)                return results# Package init__all__ = [    'precision_at_k', 'recall_at_k', 'ndcg_at_k',    'catalog_coverage', 'intra_list_diversity',    'RecommenderEvaluator']

## 9. Initialize and Test the System

In [ ]:
# Initialize systemfrom data_loader import load_all_datacourses, users, interactions, skills_taxonomy, career_paths = load_all_data()# Create hybrid recommenderrecommender = HybridRecommender(courses, users, interactions, career_paths, skills_taxonomy)# Test with a useruser_id = "user_001"result = recommender.recommend(user_id, top_k=5)print(f"Recommendations for {users[user_id].name}:")print(f"Target Role: {result.target_role}")print(f"Top Courses:")for rec in result.recommendations:    print(f"  {rec.rank}. {rec.course.title} (Score: {rec.score:.2f})")    for reason in rec.reasons[:2]:        print(f"     - {reason}")

## 10. Visualize Skill Gaps

In [ ]:
import plotly.graph_objects as go# Get skill gapsskill_gaps = recommender.get_skill_gaps(user_id)all_gaps = skill_gaps.get("critical", []) + skill_gaps.get("important", [])[:5]if all_gaps:    skill_names = [g.skill_name for g in all_gaps]    current = [g.current_level for g in all_gaps]    required = [g.required_level for g in all_gaps]        fig = go.Figure()    fig.add_trace(go.Scatterpolar(r=current + [current[0]], theta=skill_names + [skill_names[0]], fill="toself", name="Current"))    fig.add_trace(go.Scatterpolar(r=required + [required[0]], theta=skill_names + [skill_names[0]], fill="toself", name="Required", opacity=0.5))    fig.update_layout(polar=dict(radialaxis=dict(range=[0,5])), title="Skill Gap Analysis")    fig.show()